In [0]:
%run ../0-common/env-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"

silver_table = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
drivers_df = spark.read.table(bronze_table)

In [0]:
from pyspark.sql import functions as F

In [0]:
drivers_selected_df = drivers_df.drop(F.col("url"))

In [0]:
drivers_renamed_df = (
    drivers_selected_df
    .withColumnsRenamed({
        "driverId":"driver_id",
        "dateOfBirth":"date_of_birth"
    })
)

In [0]:
drivers_concat_df = (
    drivers_renamed_df
    .withColumn("driver_name",
                 F.concat_ws(" ", F.initcap("name.givenName"),
                              F.initcap("name.familyName")
                            ))
    .drop(F.col("name"))
)

In [0]:
drivers_distinct_df = drivers_concat_df.dropDuplicates(["driver_id"])

In [0]:
drivers_final_df = (
    drivers_distinct_df
    .withColumn("nationality", F.initcap("nationality"))
)

In [0]:
(
    drivers_final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)